# Preprocessing: SA-AKI original-model phase

Four labeled datasets are built from the wide-format extract produced by
`sql/feature_extraction.sql` (one row per stay_id and hour_bucket, where
hour_bucket counts hours since sepsis onset): B1 static aggregates, B2 hourly
sequences, B3 non-overlapping 24-hour windows, and B4 static trend features.
B4 is a controlled comparison with B1, sharing its cohort, observation window
and labels, rather than a separate model family.

Part A cleans the data once. Part B reshapes the cleaned result into the four
representations and labels them.

Not included: train/test split, cross-validation, Tier 2 Winsorization,
median imputation, scaling and one-hot encoding. These are fitted on training
data only, inside each model training notebook, using
`src/preprocessing_fit_transform.py`. That module also contains a SMOTE
utility, but SMOTE was not used in any reported experiment.

## Setup

Imports and file paths. `data/` is gitignored.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from src.evaluation_utils import SERIES_COLORS

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

DATA_DIR = Path("../data")
FIGURES_DIR = Path("../figures")
REF_PATH = Path("../src/reference/tier1_outlier_bounds.csv")

# raw extract from sql/feature_extraction.sql
DATA_PATH = DATA_DIR / "raw_extraction.csv"

# the 22 time-varying features, in SQL column order
TIME_VARYING = [
    "heart_rate", "temperature", "spo2", "sbp", "dbp", "map",
    "anion_gap", "creatinine", "hemoglobin", "mchc", "inr", "platelets",
    "total_bilirubin", "potassium", "phosphate", "lactate", "ph",
    "pao2", "paco2", "bun", "wbc", "urine_output",
]
STATIC = ["admission_age", "gender", "admission_type"]
LABEL_TS = ["sepsis_onset_time", "extraction_end", "aki_onset_time"]
IDS = ["stay_id", "subject_id", "hour_bucket"]

WINDOW_HOURS = 24

print(f"{len(TIME_VARYING)} time-varying features")

## A1. Load the data and sanity-check it

The extract is loaded and its shape, a sample of rows, and column types are
checked. The locked cohort is 14,826 stays, so a row count near 2.2–2.3M is
expected.

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=LABEL_TS)

print("shape:", df.shape)
print("stays:", df["stay_id"].nunique())
print("patients:", df["subject_id"].nunique())
df.head()

In [ ]:
# row count check
n_rows = len(df)
if 2_200_000 <= n_rows <= 2_300_000:
    print(f"OK  {n_rows:,} rows is in the expected 2.2-2.3M range")
else:
    print(f"CHECK  {n_rows:,} rows is OUTSIDE the expected 2.2-2.3M range")

df.dtypes

### Confirm the columns match what is expected

Actual columns are compared against the expected list, so a renamed or
missing column is caught here rather than later.

In [ ]:
expected = set(IDS + TIME_VARYING + STATIC + LABEL_TS)
actual = set(df.columns)

missing = sorted(expected - actual)
extra = sorted(actual - expected)

print("missing:", missing if missing else "none")
print("extra:  ", extra if extra else "none")

assert not missing, f"extract is missing expected columns: {missing}"

# hour_bucket: whole hours since onset, starting at 0
print("\nhour_bucket dtype:", df["hour_bucket"].dtype)
print("hour_bucket min/max:", df["hour_bucket"].min(), "/", df["hour_bucket"].max())

## A2. Keep only each patient's first ICU stay

The earliest stay per subject_id is kept, by sepsis_onset_time. All other
stays for that patient are dropped entirely, including every hourly row.

In [ ]:
# one row per stay
stay_level = (
    df.groupby("stay_id", as_index=False)
      .agg(subject_id=("subject_id", "first"),
           sepsis_onset_time=("sepsis_onset_time", "first"))
)

# earliest onset per subject; stay_id breaks ties
keep_ids = (
    stay_level.sort_values(["subject_id", "sepsis_onset_time", "stay_id"])
              .groupby("subject_id", as_index=False)
              .head(1)["stay_id"]
)

before_rows, before_stays = len(df), df["stay_id"].nunique()
df = df[df["stay_id"].isin(set(keep_ids))].copy()

print(f"stays: {before_stays:,} -> {df['stay_id'].nunique():,}")
print(f"rows:  {before_rows:,} -> {len(df):,}")
print(f"one stay per patient: {df['stay_id'].nunique() == df['subject_id'].nunique()}")

## A3. Reindex onto a complete hourly grid

Each stay is given a row for every hour from 0 up to its true final hour,
computed from sepsis_onset_time and extraction_end - not from which hours
happen to have a measurement recorded. Hours with no data are filled with
NaN, not omitted.

No data is invented; existing gaps are made explicit for A4 to measure.

In [ ]:
# per-stay hour range: 0 to that stay's true final hour
stay_bounds = df.groupby("stay_id")[["sepsis_onset_time", "extraction_end"]].first()
true_max_hour = (
    (stay_bounds["extraction_end"] - stay_bounds["sepsis_onset_time"]).dt.total_seconds() // 3600
).astype(int) - 1

max_hour = true_max_hour
counts = (max_hour.values + 1).astype(int)

grid = pd.DataFrame({
    "stay_id": np.repeat(max_hour.index.values, counts),
    "hour_bucket": np.concatenate([np.arange(c) for c in counts]),
})

print(f"grid rows: {len(grid):,}  (observed rows: {len(df):,})")
print(f"empty hours introduced: {len(grid) - len(df):,}")

In [ ]:
# stay-level constants, re-attached after the merge
stay_static = (
    df.groupby("stay_id", as_index=False)[["subject_id"] + STATIC + LABEL_TS]
      .first()
)

df = (
    grid.merge(df[["stay_id", "hour_bucket"] + TIME_VARYING],
               on=["stay_id", "hour_bucket"], how="left")
        .merge(stay_static, on="stay_id", how="left")
        .sort_values(["stay_id", "hour_bucket"])
        .reset_index(drop=True)
)

print("shape after reindex:", df.shape)
assert df[STATIC].isna().sum().sum() == 0, "static columns should be complete"
df.head()

## A4. Missingness indicators and time since last measurement

Following the MIMIC-Extract approach (Wang et al., 2020), two companion
columns are added per feature:

- `{feature}_was_measured`: 1 if a real value exists this hour, 0 otherwise
- `{feature}_hours_since_measured`: hours since that feature's last real
  measurement

Both are built before forward-filling, so they reflect genuine missingness.
Hours before a feature's first measurement are left as NaN.

In [ ]:
was_measured = {}
hours_since = {}

hour = df["hour_bucket"]
stay = df["stay_id"]

for f in TIME_VARYING:
    measured = df[f].notna()
    was_measured[f + "_was_measured"] = measured.astype("int8")

    # hour of the most recent measurement, carried forward within the stay
    last_measured_hour = hour.where(measured).groupby(stay).ffill()
    hours_since[f + "_hours_since_measured"] = hour - last_measured_hour

df = pd.concat(
    [df, pd.DataFrame(was_measured, index=df.index),
         pd.DataFrame(hours_since, index=df.index)],
    axis=1,
)

print("shape after indicators:", df.shape)

In [ ]:
# measurement frequency and staleness per feature
summary = pd.DataFrame({
    "measured_pct": [df[f + "_was_measured"].mean() * 100 for f in TIME_VARYING],
    "median_hours_since": [df[f + "_hours_since_measured"].median() for f in TIME_VARYING],
    "never_measured_pct": [df[f + "_hours_since_measured"].isna().mean() * 100 for f in TIME_VARYING],
}, index=TIME_VARYING).round(2)

summary.sort_values("measured_pct", ascending=False)

### Figure: missingness across the observation window

How often each feature is missing, hour by hour, over the first 24 hours. Taken
from the reindexed grid before any forward-filling, so it shows genuine
missingness rather than values carried forward.

This is what the indicators above encode. Features measured continuously sit
near zero; laboratory values are missing for most stays in most hours, and the
first hour is emptier than those following it.

In [ ]:
window_hours = 24
observation = df[df["hour_bucket"] < window_hours]

# percent of stays missing each feature at each hour, before forward-fill
missing_pct = (
    observation.groupby("hour_bucket")[TIME_VARYING]
               .apply(lambda hour: hour.isna().mean() * 100)
               .T
)

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(missing_pct, ax=ax, cbar_kws={"label": "percent of stays missing"})
ax.set_xlabel("hour since sepsis onset")
ax.set_ylabel("")
ax.set_title("Missing values by feature and hour, before forward-fill")
fig.tight_layout()

FIGURES_DIR.mkdir(exist_ok=True)
fig.savefig(FIGURES_DIR / "missingness_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"most complete:  {missing_pct.mean(axis=1).idxmin()} "
      f"({missing_pct.mean(axis=1).min():.1f}% missing on average)")
print(f"least complete: {missing_pct.mean(axis=1).idxmax()} "
      f"({missing_pct.mean(axis=1).max():.1f}% missing on average)")

## A5. Tier 1 outlier bounds: physiologically impossible values become NaN

Values outside physiologically possible bounds are set to NaN. Bounds are
read from `src/reference/tier1_outlier_bounds.csv`, not hardcoded.

Sources: Harutyunyan, H., Khachatrian, H., Kale, D.C., Ver Steeg, G. &
Galstyan, A. (2019). Multitask learning and benchmarking with clinical time
series data. Scientific Data, 6, Article 96
(https://github.com/YerevaNN/mimic3-benchmarks). MCHC upper bound (36 g/dL):
Zandecki, M., Genevieve, F., Gerard, J. & Godon, A. (2007). Spurious counts
and spurious results on haematology analysers: a review. Part II.
International Journal of Laboratory Hematology, 29(1), 21–41.

`inr` has no upper bound; only the lower bound is applied for that feature.

In [ ]:
bounds = pd.read_csv(REF_PATH)
print(bounds.to_string(index=False))

missing_bounds = sorted(set(TIME_VARYING) - set(bounds["feature"]))
print("\nfeatures with no bounds row:", missing_bounds if missing_bounds else "none")

### Figure: Tier 1 bounds against the observed values

Where the bounds sit relative to the data they are applied to, for creatinine
and heart rate. Values are the raw ones, before any bounding. The vertical lines
mark the low and high bounds loaded above.

A bound placed far into the tail removes recording errors without touching the
clinical range.

In [ ]:
def plot_bounds(feature):
    """Histogram of raw values for one feature, with its Tier 1 bounds marked."""
    row = bounds[bounds["feature"] == feature].iloc[0]
    values = df[feature].dropna()

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.hist(values, bins=100, color=SERIES_COLORS[0])
    ax.set_yscale("log")

    ax.axvline(row["low"], color="red", linestyle="--",
               label=f"low bound ({row['low']:g})")
    if pd.notna(row["high"]):
        ax.axvline(row["high"], color="red", linestyle="--",
                   label=f"high bound ({row['high']:g})")

    ax.set_xlabel(feature)
    ax.set_ylabel("count (log scale)")
    ax.set_title(f"Tier 1 bounds against observed {feature} values")
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
    fig.tight_layout()

    FIGURES_DIR.mkdir(exist_ok=True)
    fig.savefig(FIGURES_DIR / f"outlier_bounds_{feature}.png", dpi=200,
                bbox_inches="tight")
    plt.show()

    outside = int((values < row["low"]).sum())
    if pd.notna(row["high"]):
        outside += int((values > row["high"]).sum())
    print(f"{feature}: {len(values):,} observed values, {outside:,} outside the bounds "
          f"({outside / len(values) * 100:.4f}%)")


plot_bounds("creatinine")

In [ ]:
plot_bounds("heart_rate")

In [ ]:
n_nulled = {}

for row in bounds.itertuples(index=False):
    f = row.feature
    if f not in df.columns:
        continue

    below = df[f] < row.low if pd.notna(row.low) else pd.Series(False, index=df.index)

    # a blank `high` means no upper bound
    if pd.notna(row.high):
        above = df[f] > row.high
    else:
        above = pd.Series(False, index=df.index)
        print(f"{f}: no upper bound in reference file - lower bound only")

    impossible = below | above
    n_nulled[f] = int(impossible.sum())
    df.loc[impossible, f] = np.nan

In [ ]:
# values removed per feature
effect = pd.Series(n_nulled, name="values_set_to_nan").sort_values(ascending=False)
effect = effect.to_frame()
effect["pct_of_observed"] = (
    effect["values_set_to_nan"]
    / pd.Series({f: int(df[f + "_was_measured"].sum()) for f in effect.index})
    * 100
).round(4)

print(f"total values set to NaN: {effect['values_set_to_nan'].sum():,}")
effect

> Note: A4's indicators are built before A5's bounds are applied, so a value
> later rejected by A5 still counts as `was_measured = 1` - the indicator
> reflects that a reading was taken, independent of whether the value was
> usable.

## A6. Forward-fill within each stay

Each feature is carried forward within its own stay only. Leading NaNs,
before a feature's first measurement, are left as NaN rather than
back-filled.

In [ ]:
before_na = df[TIME_VARYING].isna().sum().sum()
df[TIME_VARYING] = df.groupby("stay_id")[TIME_VARYING].ffill()
after_na = df[TIME_VARYING].isna().sum().sum()

print(f"NaNs in feature columns: {before_na:,} -> {after_na:,}")
print(f"filled: {before_na - after_na:,}")
print("\nremaining NaNs are leading gaps, before each feature's first reading")

### Figure: cohort retention at 24 and 48 hours

How much of the cohort survives a 24-hour observation window against a 48-hour
one, recomputed here from the cohort rather than quoted, so the figures reflect
the data as it currently stands.

A stay is retained if its extraction period covers the whole window. The longer
window buys more observation time at the cost of stays, which is the trade the
24-hour choice settles.

In [ ]:
stay_span = df.groupby("stay_id")[["sepsis_onset_time", "extraction_end"]].first()
span_hours = ((stay_span["extraction_end"] - stay_span["sepsis_onset_time"])
              .dt.total_seconds() / 3600)

candidate_windows = [24, 48]
retention = {w: float((span_hours >= w).mean() * 100) for w in candidate_windows}
retained_n = {w: int((span_hours >= w).sum()) for w in candidate_windows}
total = len(span_hours)

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar([f"{w}h window" for w in candidate_windows],
              [retention[w] for w in candidate_windows],
              color=SERIES_COLORS[:len(candidate_windows)])
ax.bar_label(bars, fmt="%.1f%%")
ax.set_ylabel("percent of stays retained")
ax.set_ylim(0, 105)
ax.set_title("Cohort retention by observation window length")
fig.tight_layout()

FIGURES_DIR.mkdir(exist_ok=True)
fig.savefig(FIGURES_DIR / "window_retention_comparison.png", dpi=200,
            bbox_inches="tight")
plt.show()

for w in candidate_windows:
    print(f"{w}h window: {retained_n[w]:,} of {total:,} stays retained "
          f"({retention[w]:.1f}%)")
print(f"cost of moving from 24h to 48h: "
      f"{retained_n[24] - retained_n[48]:,} stays "
      f"({retention[24] - retention[48]:.1f} percentage points)")

Part A is complete: one cleaned, gap-explicit, forward-filled row per
stay-hour. Part B reshapes this table; no further cleaning occurs.

# Part B: four representations

The cleaned table feeds four datasets. All use 24-hour windows, but they
differ in shape and in how windows are chosen and labeled:

| | representation | shape | label | model |
|---|---|---|---|---|
| B1 | static aggregates | one row per stay | stay-level | static FFNN, XGBoost |
| B2 | sequence | 24 rows per stay | stay-level | GRU |
| B3 | sliding windows | several rows per stay | window-level | temporal-feature FFNN |
| B4 | static trend features | one row per stay | stay-level | static FFNN (trend features) |

B1, B2 and B4 use the first 24 hours after sepsis onset and the single-window
rule below. B3 uses several windows per stay and a window-level rule,
described in B3.

## Labeling rule (B1, B2, B4)

For a window starting at hour `s`, `window_end = s + 24`. Comparisons are
strict (`>`, never `>=`).

- Positive: AKI occurs strictly after `window_end`.
- Negative: AKI never occurs (`aki_onset_time` is null).
- Excluded: AKI occurs at or before `window_end`. Dropped entirely.

B3 differs: a patient may contribute several windows. For a patient who
develops AKI, only the last window ending strictly before AKI is positive;
earlier windows are negative.

In [ ]:
# AKI onset as whole hours since sepsis onset, truncated, matching the
# SQL's TIMESTAMP_DIFF(..., HOUR)
stay_tbl = df.groupby("stay_id", as_index=False)[["subject_id"] + STATIC + LABEL_TS].first()

delta = stay_tbl["aki_onset_time"] - stay_tbl["sepsis_onset_time"]
stay_tbl["aki_hour"] = np.floor(delta.dt.total_seconds() / 3600)

print(f"stays: {len(stay_tbl):,}")
print(f"never developed AKI: {stay_tbl['aki_hour'].isna().sum():,}")
print(f"developed AKI:       {stay_tbl['aki_hour'].notna().sum():,}")
stay_tbl["aki_hour"].describe()

## Verify the 24-hour guarantee

The cohort's SQL filter requires at least 24 hours between sepsis onset and
extraction_end. This is re-checked directly from the timestamps rather than
assumed, and any stay that does not actually satisfy it is dropped before
Part B continues.

In [ ]:
true_gap_hours = (stay_tbl["extraction_end"] - stay_tbl["sepsis_onset_time"]).dt.total_seconds() / 3600
fails_guarantee = stay_tbl[true_gap_hours < 24]

print(f"stays failing the true >=24h guarantee: {len(fails_guarantee):,} of {len(stay_tbl):,}")
if len(fails_guarantee) > 0:
    print("dropping these before any representation is built")
    stay_tbl = stay_tbl[~stay_tbl["stay_id"].isin(fails_guarantee["stay_id"])].copy()

In [ ]:
def label_single_window(stay_tbl, window_end=WINDOW_HOURS):
    """Label the single-window models (B1, B2).

    Returns the stay table with `label` added and excluded stays removed.
    """
    t = stay_tbl.copy()
    never_aki = t["aki_hour"].isna()
    after_window = t["aki_hour"] > window_end          # strictly after
    leaked = t["aki_hour"].notna() & ~after_window     # at or before -> excluded

    t["label"] = np.where(never_aki, 0, np.where(after_window, 1, np.nan))
    kept = t[~leaked].copy()
    kept["label"] = kept["label"].astype("int8")

    print(f"excluded (AKI at or before hour {window_end}): {leaked.sum():,}")
    print(f"kept: {len(kept):,}  positives: {kept['label'].sum():,} "
          f"({kept['label'].mean() * 100:.2f}%)")
    return kept


labeled_stays = label_single_window(stay_tbl)

## B1. Static representation: for the static FFNN and XGBoost

Each stay becomes one row. The first 24 hours of each feature are collapsed
into four numbers: min, max, mean, and last (the value at hour 23, after
forward-fill).

Static features and the label are attached; excluded stays are dropped.

In [ ]:
first_window = df[df["hour_bucket"] < WINDOW_HOURS]
print(f"rows in hours 0-23: {len(first_window):,}")

agg = first_window.groupby("stay_id")[TIME_VARYING].agg(["min", "max", "mean", "last"])
agg.columns = [f"{feat}_{stat}" for feat, stat in agg.columns]
agg = agg.reset_index()

print("aggregate columns:", agg.shape[1] - 1)
agg.head()

In [ ]:
static_representation = labeled_stays[
    ["stay_id", "subject_id"] + STATIC + ["aki_hour", "label"]
].merge(
    agg, on="stay_id", how="inner"
)

print("static representation:", static_representation.shape)
print(f"stays: {static_representation['stay_id'].nunique():,}")
print(f"positives: {static_representation['label'].sum():,} "
      f"({static_representation['label'].mean() * 100:.2f}%)")
static_representation.head()

## B2. Sequence representation: for the GRU

The same 24 hours and label as B1, kept as a 24-row-per-stay sequence rather
than summarised.

Static features are kept separate, one row per stay, routed through their
own branch at model-build time rather than repeated across all 24 rows.

Saved as `.npz`, since a 3-D array does not fit a flat CSV.

In [ ]:
seq_ids = labeled_stays["stay_id"].to_numpy()
seq_source = (
    df[df["stay_id"].isin(set(seq_ids)) & (df["hour_bucket"] < WINDOW_HOURS)]
      .sort_values(["stay_id", "hour_bucket"])
)

# drop stays with fewer than 24 hours
hours_per_stay = seq_source.groupby("stay_id").size()
short = hours_per_stay[hours_per_stay < WINDOW_HOURS]
print(f"stays with fewer than {WINDOW_HOURS} hours: {len(short):,}")

complete = hours_per_stay[hours_per_stay == WINDOW_HOURS].index
seq_source = seq_source[seq_source["stay_id"].isin(set(complete))]
print(f"stays kept for the sequence set: {seq_source['stay_id'].nunique():,}")

In [ ]:
seq_stay_order = np.sort(seq_source["stay_id"].unique())
n_stays, n_feat = len(seq_stay_order), len(TIME_VARYING)

sequences = (
    seq_source.set_index(["stay_id", "hour_bucket"])[TIME_VARYING]
              .to_numpy()
              .reshape(n_stays, WINDOW_HOURS, n_feat)
)

seq_static = (
    labeled_stays.set_index("stay_id")
                 .loc[seq_stay_order, ["subject_id"] + STATIC + ["aki_hour", "label"]]
                 .reset_index()
)

print(f"sequences: {sequences.shape}  (stays x hours x features)")
print(f"static rows: {seq_static.shape}")
print(f"positives: {seq_static['label'].sum():,} ({seq_static['label'].mean() * 100:.2f}%)")

## B3. Sliding-window representation: for the temporal-feature FFNN

Runs on the full deduplicated cohort. Each stay is divided into
non-overlapping 24-hour candidate windows: hours 0–23, then 24–47, and so on.
Only complete 24-hour windows are kept as candidates.

Candidate windows are then filtered and labeled:

- A window is eligible to be positive only if AKI occurs strictly after that
  window's own end. For a patient who develops AKI, the last eligible window
  is labeled positive and earlier windows are labeled negative. Later windows
  are excluded, and a patient with no eligible window contributes no rows.
- A patient who never develops AKI keeps every complete window, all labeled
  negative.

All candidate windows are generated first; ineligible ones are removed
afterwards rather than prevented from being generated.

Each feature gets three numbers per window: first, last, and trend
(`last - first`).

`subject_id` is retained for cluster-aware evaluation, since windows from
one patient are not independent.

In [ ]:
w = df.copy()
w["window_id"] = w["hour_bucket"] // WINDOW_HOURS

# keep only complete 24-hour windows
hours_in_window = w.groupby(["stay_id", "window_id"]).size().rename("n_hours")
complete_windows = hours_in_window[hours_in_window == WINDOW_HOURS].reset_index()[["stay_id", "window_id"]]

w = w.merge(complete_windows, on=["stay_id", "window_id"], how="inner")
print(f"complete windows: {len(complete_windows):,} across {complete_windows['stay_id'].nunique():,} stays")

In [ ]:
g = w.groupby(["stay_id", "window_id"])
first_last = g[TIME_VARYING].agg(["first", "last"])
first_last.columns = [f"{feat}_{stat}" for feat, stat in first_last.columns]
first_last = first_last.reset_index()

# trend = last - first
for f in TIME_VARYING:
    first_last[f + "_trend"] = first_last[f + "_last"] - first_last[f + "_first"]

print("engineered columns per window:", first_last.shape[1] - 2)

In [ ]:
windows = first_last.merge(
    stay_tbl[["stay_id", "subject_id", "aki_hour"] + STATIC], on="stay_id", how="left"
)
windows["window_end"] = windows["window_id"] * WINDOW_HOURS + WINDOW_HOURS

# a window is eligible to be the positive one only if AKI falls strictly
# after its own end
windows["clean_and_before_aki"] = windows["aki_hour"].notna() & (windows["aki_hour"] > windows["window_end"])

# the last eligible window is the positive one
last_valid = (
    windows[windows["clean_and_before_aki"]].groupby("stay_id")["window_id"].max().rename("stop_window")
)
windows = windows.merge(last_valid, on="stay_id", how="left")

# AKI patients: keep windows up to and including the positive one, and no
# windows at all if none is eligible. Non-AKI patients: keep everything.
has_aki = windows["aki_hour"].notna()
keep = (~has_aki) | (windows["stop_window"].notna() & (windows["window_id"] <= windows["stop_window"]))
windows = windows[keep].copy()
windows["label"] = ((windows["window_id"] == windows["stop_window"]) & has_aki).astype("int8")

print(f"windows kept: {len(windows):,}")
print(f"positives: {windows['label'].sum():,} ({windows['label'].mean() * 100:.2f}%)")
print(f"stays contributing: {windows['stay_id'].nunique():,}")
print(f"windows per stay: mean {windows.groupby('stay_id').size().mean():.2f}, "
      f"max {windows.groupby('stay_id').size().max()}")

In [ ]:
feature_cols = [f"{f}_{s}" for f in TIME_VARYING for s in ("first", "last", "trend")]
sliding_window_representation = windows[
    ["stay_id", "subject_id", "window_id"] + STATIC + feature_cols + ["label"]
].reset_index(drop=True)

print("sliding-window representation:", sliding_window_representation.shape)
sliding_window_representation.head()

## B4. Static representation with trend features

The same cohort, the same single 24-hour window and the same stay-level label as
B1, with only the time-varying feature columns changed: first value in the
window, last value in the window, and the difference between them, matching the
trend calculation used in B3.

This differs from both existing representations. B1 summarises the window as
min, max, mean and last. B3 uses first, last and trend but over several sliding
windows with a window-level label. B4 applies B3's feature engineering to B1's
single window and label, so feature representation is the only thing that
differs between B1 and B4.

The walk-forward window logic from B3 is not used here. Every stay contributes
exactly one row, as in B1.

In [ ]:
# first and last within the same hours 0-23 window B1 uses
trend_agg = first_window.groupby("stay_id")[TIME_VARYING].agg(["first", "last"])
trend_agg.columns = [f"{feat}_{stat}" for feat, stat in trend_agg.columns]
trend_agg = trend_agg.reset_index()

# trend = last - first, the same calculation B3 applies
for f in TIME_VARYING:
    trend_agg[f + "_trend"] = trend_agg[f + "_last"] - trend_agg[f + "_first"]

print("trend feature columns:", trend_agg.shape[1] - 1)
trend_agg.head()

In [ ]:
static_trend_representation = labeled_stays[
    ["stay_id", "subject_id"] + STATIC + ["aki_hour", "label"]
].merge(trend_agg, on="stay_id", how="inner")

print("static trend representation:", static_trend_representation.shape)
print(f"stays: {static_trend_representation['stay_id'].nunique():,}")
print(f"positives: {static_trend_representation['label'].sum():,} "
      f"({static_trend_representation['label'].mean() * 100:.2f}%)")
static_trend_representation.head()

### Verification against B1

B4 is built from the identical cohort, window and label definition, so its row
count, contributing patient count and positive rate must match B1 exactly. A
mismatch means the two have diverged somewhere they should not have, and is
treated as an error rather than reported and passed over.

In [ ]:
checks = {
    "rows": (len(static_representation), len(static_trend_representation)),
    "stays": (static_representation["stay_id"].nunique(),
              static_trend_representation["stay_id"].nunique()),
    "patients": (static_representation["subject_id"].nunique(),
                 static_trend_representation["subject_id"].nunique()),
    "positives": (int(static_representation["label"].sum()),
                  int(static_trend_representation["label"].sum())),
}

for name, (b1, b4) in checks.items():
    print(f"  {name:10s} B1 {b1:>7,}   B4 {b4:>7,}   {'match' if b1 == b4 else 'MISMATCH'}")

rate_b1 = static_representation["label"].mean()
rate_b4 = static_trend_representation["label"].mean()
print(f"  {'positive rate':10s} B1 {rate_b1 * 100:.4f}%   B4 {rate_b4 * 100:.4f}%")

mismatched = [name for name, (b1, b4) in checks.items() if b1 != b4]
assert not mismatched, f"B4 does not match B1 on: {mismatched}"
assert rate_b1 == rate_b4, "B4 positive rate differs from B1"

# the same stays, in the same order, not merely the same counts
assert (static_representation["stay_id"].to_numpy()
        == static_trend_representation["stay_id"].to_numpy()).all(), \
    "B4 holds different stays from B1"

print()
print("B4 matches B1 on cohort, window and label; only the feature columns differ")
print(f"  B1 feature columns: {static_representation.shape[1] - 6}")
print(f"  B4 feature columns: {static_trend_representation.shape[1] - 6}")

## Save the four datasets

Written to `data/`, which is gitignored.

Tabular sets are saved as Parquet. The sequence set is saved as `.npz`, with
its per-stay static table saved separately as Parquet.

In [ ]:
DATA_DIR.mkdir(exist_ok=True)

static_representation.to_parquet(DATA_DIR / "static_representation.parquet", index=False)
sliding_window_representation.to_parquet(DATA_DIR / "sliding_window_representation.parquet", index=False)
static_trend_representation.to_parquet(DATA_DIR / "static_trend_representation.parquet", index=False)

np.savez_compressed(
    DATA_DIR / "sequence_representation.npz",
    sequences=sequences,
    stay_id=seq_stay_order,
    feature_names=np.array(TIME_VARYING),
)
seq_static.to_parquet(DATA_DIR / "sequence_static.parquet", index=False)

for p in sorted(DATA_DIR.glob("*representation*")) + [DATA_DIR / "sequence_static.parquet"]:
    print(f"{p.name:45s} {p.stat().st_size / 1e6:8.1f} MB")

## Summary and sanity check

Final counts and class balance are reported for the four outputs, as a
check that labeling behaved as expected.

B1, B2 and B4 are expected to match exactly in stay count and positive rate,
since all three apply the identical single-window rule to the same 24-hour
block. B1 and B4 share the same patients, observation window and labels;
only their feature representation differs, as asserted in the verification
against B1 above.

B3 is expected to have more rows than stays, since a patient may contribute
several windows, and a lower positive rate, since only one window per
AKI-positive patient is positive.

In [ ]:
rows = []
rows.append({
    "representation": "B1 static",
    "rows": len(static_representation),
    "stays": static_representation["stay_id"].nunique(),
    "positives": int(static_representation["label"].sum()),
    "positive_rate_%": round(static_representation["label"].mean() * 100, 2),
})
rows.append({
    "representation": "B2 sequence",
    "rows": int(sequences.shape[0]),
    "stays": int(sequences.shape[0]),
    "positives": int(seq_static["label"].sum()),
    "positive_rate_%": round(seq_static["label"].mean() * 100, 2),
})
rows.append({
    "representation": "B3 sliding",
    "rows": len(sliding_window_representation),
    "stays": sliding_window_representation["stay_id"].nunique(),
    "positives": int(sliding_window_representation["label"].sum()),
    "positive_rate_%": round(sliding_window_representation["label"].mean() * 100, 2),
})

rows.append({
    "representation": "B4 static trend",
    "rows": len(static_trend_representation),
    "stays": static_trend_representation["stay_id"].nunique(),
    "positives": int(static_trend_representation["label"].sum()),
    "positive_rate_%": round(static_trend_representation["label"].mean() * 100, 2),
})

pd.DataFrame(rows)

In [ ]:
print(f"sequence array: {sequences.shape}  (stays x {WINDOW_HOURS} hours x {len(TIME_VARYING)} features)")
print(f"B3 windows per stay: mean "
      f"{len(sliding_window_representation) / sliding_window_representation['stay_id'].nunique():.2f}")
print("\nRemaining NaNs are leading gaps only - no median imputation, scaling,")
print("encoding, Winsorization or SMOTE has been applied. All of that is fit on")
print("training data only and belongs in the training-setup notebook.")